# Curanex: colorectal cancer in pathology tiles - threshold set on CRC-VAL, tested on NCT-CRC-HE-100K (CPU)

The Kather-2016-trained model ranked CRC-VAL-HE-7K tumour tiles very well (AUC 0.977) but its validation threshold caught 11%.
**Rules fixed before running (2026-11-12):** model unchanged; tumour score = P(tumour); threshold = highest with sensitivity >= 0.85
on CRC-VAL-HE-7K (already used once, now the calibration set). **Outside test, scored once:** NCT-CRC-HE-100K (86 other slides,
NCT Biobank / UMM pathology archive), tumour (TUM) vs every other tissue class. Verified if sensitivity and AUC are both >= 0.80.

In [ ]:
!pip install -q onnxruntime

In [ ]:
import glob, json, os
import numpy as np, onnxruntime as ort
from PIL import Image
from sklearn.metrics import roc_auc_score
meta = json.load(open(glob.glob("/kaggle/input/**/crc_ft.json", recursive=True)[0]))
sess = ort.InferenceSession(glob.glob("/kaggle/input/**/crc_ft.onnx", recursive=True)[0])
MEAN, STD = np.array(meta["mean"], np.float32), np.array(meta["std"], np.float32)


def prep(f):
    im = Image.open(f).convert("RGB"); w, h = im.size; s = 256 / min(w, h)
    im = im.resize((max(224, round(w * s)), max(224, round(h * s))), Image.BILINEAR); w, h = im.size
    l, t = (w - 224) // 2, (h - 224) // 2
    return ((np.asarray(im.crop((l, t, l + 224, t + 224)), np.float32) / 255 - MEAN) / STD).transpose(2, 0, 1)


def scores(files):
    out = []
    for i in range(0, len(files), 128):
        out += list(sess.run(None, {"image": np.stack([prep(f) for f in files[i:i + 128]])})[0][:, 0])
    return np.array(out)


def report(y, p, t):
    y, p = np.array(y), np.array(p); f = p >= t
    r = {"n": int(len(y)), "positives": int(y.sum()), "sensitivity": round(float(f[y == 1].mean()), 3),
         "specificity": round(float((~f[y == 0]).mean()), 3), "auc": round(float(roc_auc_score(y, p)), 3)}
    low = min(r["sensitivity"], r["auc"]); r["status"] = "verified" if low >= 0.80 else "live" if low >= 0.75 else "off"
    return r


tif = lambda pat: sorted(glob.glob(pat, recursive=True))
val = tif("/kaggle/input/**/CRC-VAL-HE-7K/**/*.tif")
nct = [f for f in tif("/kaggle/input/**/NCT-CRC-HE-100K/**/*.tif") if "VAL" not in f.upper()]
lab = lambda f: int(os.path.basename(os.path.dirname(f)).upper() == "TUM")
print("CRC-VAL:", len(val), sum(map(lab, val)), "| NCT-100K:", len(nct), sum(map(lab, nct)))
assert val and nct
pv = scores(val); yv = np.array([lab(f) for f in val])
T = float(max(v for v in np.unique(np.round(pv, 6)) if (pv[yv == 1] >= v).mean() >= 0.85))
print("threshold", T, "calibration (CRC-VAL):", report(yv, pv, T), flush=True)
res = report([lab(f) for f in nct], scores(nct), T)
print("OUTSIDE TEST on NCT-CRC-HE-100K (once):", res)
json.dump({"threshold": T, "outside": res}, open("/kaggle/working/crc_nct.json", "w"), indent=1)